In [3]:
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class SimpleCNN(nn.Module):
    def __init__(self, in_channels=3, feature_dim=256):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(256, feature_dim)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


class ImageClassifier(nn.Module):
    def __init__(self, feature_dim=256):
        super().__init__()
        self.backbone = SimpleCNN(in_channels=3, feature_dim=feature_dim)
        self.head = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Linear(feature_dim, 1)
        )

    def forward(self, x):
        feat = self.backbone(x)
        logit = self.head(feat)
        return torch.sigmoid(logit).squeeze(1)


In [5]:
import os
import cv2
import torch
import torch.nn as nn
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- utils ----------
def detect_file_type(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        return "image"
    elif ext in [".mp4", ".avi", ".mov", ".mkv"]:
        return "video"
    return "invalid"


img_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])


def preprocess_single_image(path):
    path = os.path.abspath(path)

    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    img = cv2.imread(path)
    if img is None:
        raise RuntimeError(f"OpenCV failed to read image: {path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img_transform(img)
    return img.unsqueeze(0).to(DEVICE)


def unified_predict(path):
    ftype = detect_file_type(path)

    if ftype != "image":
        raise RuntimeError("Only image inference is enabled right now.")

    x = preprocess_single_image(path)
    with torch.no_grad():
        prob_fake = model(x).item()

    return {
        "file_type": "image",
        "fake_probability": prob_fake,
        "real_probability": 1.0 - prob_fake
    }


In [7]:
result = unified_predict(r"test_images\TEST1.png")
print(result)

FileNotFoundError: File not found: b:\deepfake_detection\deepfake_dataset\test_images\TEST1.png